In [0]:
import numpy as np
import pandas as pd

df = spark.sql("""
with items as (
  select
    i.order_id,
    max(i.distance_km)                    as max_distance_km,
    sum(p.weight_g)                       as total_weight_g,
    sum(p.volume_cm3)                     as total_volume_cm3,
    max_by(s.state, i.distance_km)        as seller_state,
    max_by(p.category, i.price)           as main_category
  from olist.gold.fact_order_items i
  join olist.gold.dim_product p on i.product_id = p.product_id
  join olist.gold.dim_seller  s on i.seller_id  = s.seller_id
  group by i.order_id
)
select
  o.order_id,
  cast(o.is_late as int)                  as is_late,
  o.promised_days,
  o.item_count,
  o.seller_count,
  cast(o.items_total as double)           as items_total,
  cast(o.freight_total as double)         as freight_total,
  o.max_installments,
  o.main_payment_type,
  month(o.purchased_at)                   as purchase_month,
  dayofweek(o.purchased_at)               as purchase_dow,
  c.state                                 as customer_state,
  it.seller_state,
  cast(c.state = it.seller_state as int)  as same_state,
  it.max_distance_km,
  it.total_weight_g,
  it.total_volume_cm3,
  it.main_category
from olist.gold.fact_orders o
join olist.gold.dim_customer c on o.customer_id = c.customer_id
join items it                  on o.order_id = it.order_id
where o.order_status = 'delivered'
  and o.delivered_at is not null
""").toPandas()

# Engineered feature: freight as a share of the order value
df["freight_ratio"] = df["freight_total"] / df["items_total"].replace(0, np.nan)

print(f"{len(df):,} orders, late rate {df['is_late'].mean():.1%}")

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric = ["promised_days", "item_count", "seller_count", "items_total", "freight_total",
           "freight_ratio", "max_installments", "max_distance_km", "total_weight_g",
           "total_volume_cm3", "same_state"]
categorical = ["customer_state", "seller_state", "main_category", "main_payment_type",
               "purchase_month", "purchase_dow"]

df[categorical] = df[categorical].astype(str)   # treat month/day numbers as labels

X = df[numeric + categorical]
y = df["is_late"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
    ]), numeric),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=False)),
    ]), categorical),
])

In [0]:
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score)

def evaluate(model, threshold=0.5):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        "roc_auc":   roc_auc_score(y_test, proba),
        "pr_auc":    average_precision_score(y_test, proba),
        "precision": precision_score(y_test, pred),
        "recall":    recall_score(y_test, pred),
    }

models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "gradient_boosting":   HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
}

mlflow.sklearn.autolog()
results = {}
for name, estimator in models.items():
    with mlflow.start_run(run_name=name):
        pipe = Pipeline([("prep", preprocess), ("model", estimator)])
        pipe.fit(X_train, y_train)
        metrics = evaluate(pipe)
        mlflow.log_metrics({f"test_{k}": v for k, v in metrics.items()})
        results[name] = (pipe, metrics)

pd.DataFrame({name: m for name, (_, m) in results.items()}).T.round(3)

In [0]:
best_name = max(results, key=lambda n: results[n][1]["pr_auc"])
best_model = results[best_name][0]
print(f"Best model: {best_name}")

rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    m = evaluate(best_model, threshold=t)
    flagged = (best_model.predict_proba(X_test)[:, 1] >= t).mean()
    rows.append({"threshold": t, "precision": m["precision"], "recall": m["recall"],
                 "pct_orders_flagged": flagged})
pd.DataFrame(rows).round(3)

In [0]:
from sklearn.inspection import permutation_importance

sample = X_test.sample(5000, random_state=42)
imp = permutation_importance(best_model, sample, y_test.loc[sample.index],
                             scoring="average_precision", n_repeats=5, random_state=42)

pd.Series(imp.importances_mean, index=sample.columns).sort_values(ascending=False).round(4)

## Model results
- **Goal:** flag orders likely to arrive late, at purchase time (no post-purchase data used).
- **Best model:** [gradient boosting], ROC-AUC [0.789], PR-AUC [0.267] vs. 0.07 random baseline.
- **Chosen threshold:** [0.6], catching [56%] of late orders while flagging [17%] of all orders.
- **Top drivers: purchase month (seasonality), promised delivery window, and customer’s state, followed by seller-to-customer distance and whether the seller ships within the same state. Product category, weight, price, and payment details had almost no predictive value.
- **Limitations:** lateness also depends on carrier and external factors not in the data, which caps achievable accuracy